[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 Medium: Adam Optimizer

Implement the **Adam** optimizer from scratch.

### Signature
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### Algorithm (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.9 MB/s eta 0:00:00


In [4]:
import torch

In [30]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps

        self.t = 0

        # For each parameter, keep first and second moments.
        self.m = []
        self.v = []

        for p in self.params:
            self.m.append(torch.zeros_like(p))
            self.v.append(torch.zeros_like(p))

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            grad = p.grad

            # m = β1 * m + (1-β1) * grad
            self.m[i] = self.m[i] * self.beta1 + grad * (1 - self.beta1)
            # v = β2 * v + (1-β2) * grad²
            self.v[i] = self.v[i] * self.beta2 + grad ** 2 * (1 - self.beta2)

            # m̂ = m / (1 - β1ᵗ)
            # v̂ = v / (1 - β2ᵗ)
            m_ = self.m[i] / (1 - self.beta1 ** self.t)
            v_ = self.v[i] / (1 - self.beta2 ** self.t)

            with torch.no_grad():
                # p -= lr * m̂ / (√v̂ + ε)
                # self.params[i] = p - self.lr * m_ / (torch.sqrt(v_) + self.eps) # why this line not working
                # p -= self.lr * m_ / (torch.sqrt(v_) + self.eps) # why this line is working?
                p.add_(- self.lr * m_ / (torch.sqrt(v_) + self.eps)) # why this line is working?

    def zero_grad(self):
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            # p.grad.mul_(0.0)
            p.grad.zero_()

In [31]:
# 🧪 Debug
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'Step {i}: loss={loss.item():.4f}')

Step 0: loss=12.5974
Step 1: loss=12.3944
Step 2: loss=12.1939
Step 3: loss=11.9960
Step 4: loss=11.8005


In [32]:
# ✅ SUBMIT
from torch_judge import check
check('adam')


🧪 Testing: Adam Optimizer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Parameters change after step (5.7ms)
  ✅ [2/3] Matches torch.optim.Adam (6.1ms)
  ✅ [3/3] zero_grad works (0.5ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (12.3ms total)
  Progress saved. Run status() to see your dashboard.

